# §61: class prototypes on `v^t` in the main score, KIM off — **K0dfiP / K0dfiL / K0dfiPL / K0dfiPLb** vs **K0dfi**, s2024

Rule: `core/docs/final/KIM_RUN.md` **§61**, written before any number of these rows. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. `nexar_test` is never touched. **K5f stays the project reference; nothing is adopted at n = 1.** Reads: Nexar val, DoTA-CAP-dev, DADA, DAD — **no TAD** (user).

| row | score term `λ·(cos(v,μ1) − cos(v,μ0))` | normal-anchored loss | μ0 + bottom-k |
|---|---|---|---|
| K0dfi (control, §53) | – | – | – |
| **K0dfiP** | ✓ (λ init 0) | – | – |
| **K0dfiL** | – | ✓ | – |
| **K0dfiPL** (decides) | ✓ | ✓ | – |
| **K0dfiPLb** | ✓ | ✓ | ✓ |

* **Control (PR6):** K0dfi trained on an **NVIDIA L4** (§53). If this session's GPU / torch / CUDA equal its `env.json`, its checkpoint and reads are reused from `kim_ablation/`; otherwise **K0dfirr** (its exact command) is trained here first.
* **Start gate (PR7):** every new row's step-1 `mil` must equal the control's. **Live gate:** `proto_ready` = 1 on the last step (the report raises otherwise).

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (K0dfi, K5f, K5dfi runs + reads, bakes, KNN), control decision | no | 2-5 min |
| 4 | pytest on the touched code — **raises if red** | no | 3 min |
| 5 | train [K0dfirr,] 4 rows + checks | yes | 4-5 × K0dfi |
| 6 | zero-shot reads: DoTA-CAP-dev (protocol B), DADA whole, DAD | yes | 15-25 min per row |
| 7 | `nexar_val` read with `--save-scores` | yes | 10-20 min per row |
| 8 | report + `zs_arm_table` + `dad_position_diag` -> **paste `REPORTS/k0dfi_proto_summary.md` to Claude** | no | 3-5 min |

`RUN_ROWS` in step 3 can be shortened to split the training over sessions; a re-run skips what is on Drive. Thesis and Thesis-V2 are only read; everything new goes to `Thesis-Final/outputs/kim_dfi_proto/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, Nexar caches, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K0dfi / K5f / K5dfi and their reads, DAD; every output
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 61. Class prototypes on `v^t`' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), \
    'KIM_RUN.md has no §61 -- re-upload kat-vad/'
assert (REPO / 'core/models/proto_bank.py').is_file() and (REPO / 'core/tools/k0dfi_proto_report.py').is_file(), \
    'no proto_bank / k0dfi_proto_report -- re-upload kat-vad/'
assert '"proto")' in (REPO / 'core/inference.py').read_text(), 'inference.py does not adopt proto.* -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                  # sub-folder name of every eval read (eval/<set>/K5)
K5F      = constants.KIM_FIXED_ARM                 # the project reference (§31)
K5DFI    = constants.KIM_DVS_IGNORE_ROW            # KIM on, same host (printed contrast)
HOST     = constants.PROTO_HOST_ROW                # K0dfi (§53)
RR       = HOST + constants.KIM_ABLATION_RERUN_SUFFIX   # K0dfirr, only if the env differs (PR6)
ROWS     = constants.PROTO_ROWS                    # row -> its proto.* overrides (PR5)
NEW_ROWS = list(ROWS)
SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
DATASET  = constants.NEXAR_DATASET
C_WINDOW = constants.NEXAR_CONSTRUCTION_WINDOW
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # nexar_test is never read here
DAD_NAME, DAD_STRIDE, DAD_SET = constants.DAD_DATASET, constants.DAD_STRIDE, constants.DAD_OUTPUT_SET

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C_WINDOW}'     # K0dfi's corpus
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                # nexar_train statistics
FIXED    = constants.OUTPUT_ROOT / constants.KIM_FIXED_OUTPUT_DIRNAME
K5F_ROOT = FIXED / K5F
K5DFI_ROOT = constants.OUTPUT_ROOT / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / K5DFI
HOST_ROOT = constants.OUTPUT_ROOT / constants.PROTO_HOST_OUTPUT_DIRNAME / HOST   # kim_ablation/K0dfi
# Zero-shot inputs: the bakes K0dfi was read on
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_DAD   = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
# Nexar
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C_WINDOW}' / constants.KNN_CACHE_FILENAME   # DVS fillers
# Outputs
ROOT     = constants.OUTPUT_ROOT / constants.PROTO_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k0dfi_proto')
LOCAL.mkdir(parents=True, exist_ok=True)
CTRL, CTRL_ROOT = None, None                       # decided in step 3 (PR6): (HOST, HOST_ROOT) or (RR, ROOT / RR)


def row_root(row):
    return CTRL_ROOT if row == CTRL else ROOT / row


def row_paths(row):
    run_dir = row_root(row) / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', row_root(row) / 'eval'


HOST_RUN = HOST_ROOT / K5 / f's{SEED}'
INPUTS = (('Nexar_window labels', WINDOW / constants.LABELS_TRAIN_FILENAME), ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('KNN cache', KNN), ('K0dfi config', HOST_RUN / 'config.yaml'), ('K0dfi metrics', HOST_RUN / 'metrics.jsonl'),
          ('K0dfi verified', HOST_RUN / 'verified.json'), ('K0dfi env', HOST_RUN / constants.KIM_RUN_ENV_FILENAME),
          ('K5f run', K5F_ROOT / K5 / f's{SEED}' / 'metrics.jsonl'), ('K5f nexar_val read', K5F_ROOT / 'eval' / 'nexar' / K5F / 'video_stats.json'),
          ('K5dfi run', K5DFI_ROOT / K5 / f's{SEED}' / 'metrics.jsonl'), ('K5dfi nexar_val read', K5DFI_ROOT / 'eval' / 'nexar' / K5DFI / 'video_stats.json'),
          *[(f'{r} {s} read', root / 'eval' / s / K5) for r, root in ((K5F, K5F_ROOT), (K5DFI, K5DFI_ROOT)) for s in ('dota', 'dada', DAD_SET)],
          ('DADA A2 bake', A2_DADA), ('DoTA-CAP A2 bake', A2_DOTA), ('DAD A2 bake', A2_DAD),
          ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
          ('DoTA s1', DOTA_S1), ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
assert not (WINDOW / constants.FORCED_NORMALS_FILENAME).exists(), 'Nexar_window holds forced normals: not K0dfi\'s corpus'
print(f'{len(INPUTS)} inputs found | new rows {NEW_ROWS} | runs/reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 5-7 need one'
df -h /content | tail -1

In [ ]:
import torch
import yaml

from core.data.v2_splits import load_split           # noqa: E402
from core.tools import k5_edge_report as edge        # noqa: E402
from core.tools import kim_ablation_report as ka     # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON, VIDEO_STATS_JSON  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(drive_dir, steps):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


# Preflight on small files only (C39)
host_cfg = _flat(yaml.safe_load((HOST_RUN / 'config.yaml').read_text()))
assert not host_cfg['kim.enabled'] and host_cfg['dvs.theta'] == 0.7 and host_cfg['loss.dvs_anchor_mode'] == constants.DVS_ANCHOR_MODE_IGNORE, \
    'HOST_RUN is not K0dfi'
STEPS = json.loads((HOST_RUN / 'verified.json').read_text())['step']          # 1,749
EPOCHS = host_cfg['train.num_epochs']
assert metrics_rows(HOST_RUN / 'metrics.jsonl')[-1]['global_step'] == STEPS, 'K0dfi: J10'
SHA = manifest(A2_NEXAR)['train_ids_sha1']
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('DAD', A2_DAD)):
    m = manifest(d)
    assert (m['crn'], m['motion'], m['train_ids_sha1']) == (constants.V2_OFF, ENC, SHA), f'{name} bake: {m}'
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DADA_IDS) == 370 and len(DAD_IDS) == constants.DAD_TEST_VIDEOS
for name, d, ids in (('DADA bake', A2_DADA, DADA_IDS), ('DoTA-CAP bake', A2_DOTA, CAP_IDS), ('DAD bake', A2_DAD, DAD_IDS),
                     ('Nexar CLIP', NEXAR_X, NEXAR_IDS), ('Nexar V2-S', NEXAR_U, NEXAR_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
print(f'preflight OK: K0dfi step {STEPS} ({EPOCHS} epochs) | DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, '
      f'DAD {len(DAD_IDS)}, {NEXAR_SPLIT} {len(NEXAR_IDS)}')


def reads_complete(root, row):
    ev = root / 'eval'
    return all(p.exists() for p in (
        ev / 'dota' / K5 / 'clip_aucs.json', ev / 'dada' / K5 / 'clip_aucs.json',
        ev / DAD_SET / K5 / f's{SEED}' / 'results.json', ev / 'nexar' / row / READOUT_JSON,
        ev / 'nexar' / row / VIDEO_STATS_JSON))


# PR6: reuse K0dfi only on the environment it trained in, with every read the report needs
ENV_NOW = ({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': str(torch.version.cuda)}
           if torch.cuda.is_available() else None)
HOST_ENV = json.loads((HOST_RUN / constants.KIM_RUN_ENV_FILENAME).read_text())
if is_verified(ROOT / RR / K5 / f's{SEED}', STEPS):
    CTRL, CTRL_ROOT, why = RR, ROOT / RR, 'K0dfirr was re-trained by this notebook before'
elif HOST_ENV == ENV_NOW and HOST_ENV['gpu'] == constants.PROTO_HOST_GPU and is_verified(HOST_RUN, STEPS) \
        and reads_complete(HOST_ROOT, HOST):
    CTRL, CTRL_ROOT, why = HOST, HOST_ROOT, f'same environment as §53 ({ENV_NOW}): K0dfi reused'
else:
    CTRL, CTRL_ROOT, why = RR, ROOT / RR, f'this session {ENV_NOW} vs K0dfi {HOST_ENV}: K0dfirr is trained here'
RUN_ROWS = NEW_ROWS if CTRL == HOST else [RR, *NEW_ROWS]   # shorten to split the training over sessions
print(f'PR6 control: {why} -> {CTRL_ROOT} | rows to train/read: {RUN_ROWS}')

## 4. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_proto_bank.py',
     'core/tests/test_k0dfi_proto_report.py', 'core/tests/test_kim.py', 'core/tests/test_kim_input_encoder.py',
     'core/tests/test_dvs_dataset.py', 'core/tests/test_ckpt_compat.py'])

## 5. Train the rows (§61 PR5-PR8) — K0dfirr first when it is re-trained

**Watch the log: past 2.5 s/step, stop and paste it.** K0dfirr: config diff to K0dfi empty. Each new row: exactly its `proto.*` keys. Printed per row: start gate (PR7), last-step λ / `proto_cos` / `proto_ready`; the report checks again.

In [ ]:
BASE = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={DATASET}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
        'kip.enabled=false', f'loss.dvs_anchor_mode={constants.DVS_ANCHOR_MODE_IGNORE}', 'data.is_egocentric=false',
        f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}', 'dvs.theta=0.7', 'dvs.theta_ego=0.85',
        *[f'{k}=false' for k in constants.KIM_ABLATION_FLAGS]]          # = K0dfi's command (§53)
ARMS = {RR: {}, **ROWS}                                                 # row -> its proto.* overrides
L_KNN = ['--knn-cache', stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)]


def last_epoch(rows, key):
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    return sum(r.get(key, float('nan')) for r in last) / len(last)


def check_run(row):
    run_dir, ckpt, _ = row_paths(row)
    rows = metrics_rows(run_dir / 'metrics.jsonl')
    payload = torch.load(ckpt, map_location='cpu', weights_only=False)
    assert payload['global_step'] == rows[-1]['global_step'] == STEPS, f'{row}: J10'
    got = _flat(yaml.safe_load((run_dir / 'config.yaml').read_text()))
    diff = ka.config_diff(host_cfg, got)
    want = {k: v == 'true' for k, v in ARMS.get(row, {}).items()}
    assert {k: v[1] for k, v in diff.items()} == want, f'{row}: config differs from K0dfi by {diff}, §61 allows {want}'
    mul = last_epoch(rows, 'mul_mil')
    assert mul > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{row}: last-epoch mul_mil {mul:.4f} -- paste to Claude'
    msg = f'{row}: config diff {sorted(want)} | last-epoch mil {last_epoch(rows, "mil"):.4f} mul_mil {mul:.4f}'
    if row in NEW_ROWS:
        extra = {k for k in payload['model'] if k.startswith('proto.')}
        assert extra and all(k.startswith('proto.') for k in set(payload['model']) - extra), f'{row}: state dict'
        st = edge.start_check(row_paths(CTRL)[0], run_dir)
        last = rows[-1]
        assert last.get('proto_ready') == 1.0, f'{row}: proto_ready {last.get("proto_ready")} -- the prototypes never both existed'
        lam = last.get('proto_lambda')
        msg += (f"\n  PR7 start: {st['new']:.6f} vs {CTRL} {st['control']:.6f} -> "
                f"{'OK' if st['same_draws'] else 'FLAG: NOT the same start -- paste to Claude'}"
                f"\n  last step: λ {'n/a' if lam is None else f'{lam:+.4f}'} | cos(μ0, μ1) {last['proto_cos']:+.3f} | "
                f"anchor share {last['proto_anchor_share']:.3f} | proto_nkc {last.get('proto_nkc', 'n/a')}")
    print(msg)


def train_arm(row):
    run_dir, ckpt, _ = row_paths(row)
    if is_verified(run_dir, STEPS):
        print(f'{row} s{SEED}: verified on Drive -- not retrained')
        check_run(row)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / row
    local.mkdir(parents=True, exist_ok=True)
    env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': str(torch.version.cuda)}
    ctrl_env = row_paths(CTRL)[0] / constants.KIM_RUN_ENV_FILENAME
    if row != CTRL and ctrl_env.exists() and json.loads(ctrl_env.read_text()) != env:
        print(f'FLAG: {row} trains on {env}, {CTRL} on {json.loads(ctrl_env.read_text())}: the start gate will tell')
    (local / constants.KIM_RUN_ENV_FILENAME).write_text(json.dumps(env))
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    sets = [*BASE, *(f'{k}={v}' for k, v in ARMS[row].items())]
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in sets for x in ('--set', s)],
         '--data-dir', stage(WINDOW, LOCAL / 'stage' / 'data'), '--clip-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar'),
         '--output-dir', local, *L_KNN, *resume], log=LOCAL / f'train_{row}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml', constants.KIM_RUN_ENV_FILENAME):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{row}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{row}: trained in {(time.time() - t0) / 60:.0f} min')
    check_run(row)


if CTRL not in RUN_ROWS:
    check_run(CTRL)                                # the reused control: the same checks, printed
for row in RUN_ROWS:
    train_arm(row)

## 6. Zero-shot reads of every trained row: DoTA-CAP-dev (protocol B), DADA (whole), DAD — K0dfi's protocols and bakes (no TAD)

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)


def read_zero_shot(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt,
             '--input-dir', stage(A2_DOTA, LOCAL / 'stage' / 'cap', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels'), '--split', CAP_DEV,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_dota.log')
        publish(local, out)
    print(f"{row} DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")
    out = ev / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'stage' / 'dada', DADA_IDS),
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{row}_dada.log')
        publish(local, out)
    print(f"{row} DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")
    out = ev / DAD_SET / K5 / f's{SEED}'
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / row / DAD_SET
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={DAD_NAME}',
             '--data-dir', stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), '--clip-dir', stage(A2_DAD, LOCAL / 'stage' / 'dad_bake', DAD_IDS),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{row}_dad.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(DAD_IDS), r
    print(f"{row} DAD: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")
    flush_drive()


for row in RUN_ROWS:
    read_zero_shot(row)

## 7. `nexar_val` read of every trained row with `--save-scores` (`--final` is never passed)

In [ ]:
def read_nexar(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'nexar' / row
    done = all((out / f).exists() for f in (READOUT_JSON, ITEM_AUCS_JSON, VIDEO_STATS_JSON)) and \
        len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
    if not done:
        L_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
        L_U = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
        L_STATS = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])     # manifest + statistics only
        local = LOCAL / 'eval' / row / 'nexar'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', ckpt, '--stats-dir', L_STATS,
             '--video-s1-dir', L_U, '--census', CENSUS, '--clip-s1-dir', L_X,
             '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--save-scores', '--out-dir', local], log=LOCAL / f'{row}_nexar.log')
        publish(local, out)
        flush_drive()
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
    assert r['global_steps'] == {f's{SEED}': STEPS} and Path(r['checkpoints'][f's{SEED}']) == ckpt, (r['global_steps'], r['checkpoints'])
    print(f"{row}: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | "
          f"whole macro {r['whole']['macro']['mean']:.4f} | clip AUC {r['whole']['clip_auc_max']:.4f}")


for row in RUN_ROWS:
    read_nexar(row)

## 8. Report -> **paste `REPORTS/k0dfi_proto_summary.md` to Claude**

`k0dfi_proto_report` (§61: gates PR7, contrasts D / P / L / PL−P / b / R / KIM, verdict, candidacy; macro), `zs_arm_table` (control = the control row; micro / AP on DoTA-CAP-dev, DADA, DAD) and `dad_position_diag` (placebo beside every DAD number, PR10). **Every row must be complete.**

In [ ]:
def complete(row):
    run_dir, _, _ = row_paths(row)
    return is_verified(run_dir, STEPS) and reads_complete(row_root(row), row)


assert all(complete(r) for r in (CTRL, *NEW_ROWS)), 'every row must be trained and read first (RUN_ROWS shortened?)'
local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
local.mkdir(parents=True)
run([sys.executable, '-m', 'core.tools.k0dfi_proto_report', '--control-root', CTRL_ROOT, '--control-name', CTRL,
     '--root', ROOT, '--reference-root', K5F_ROOT, '--reference-nexar-dir', K5F_ROOT / 'eval' / 'nexar' / K5F,
     '--kim-root', K5DFI_ROOT, '--kim-nexar-dir', K5DFI_ROOT / 'eval' / 'nexar' / K5DFI, '--out-dir', local])
evs = [(CTRL, row_paths(CTRL)[2]), *[(r, row_paths(r)[2]) for r in NEW_ROWS],
       (K5DFI, K5DFI_ROOT / 'eval'), (K5F, K5F_ROOT / 'eval')]
reads = [a for arm, ev in evs for s in ('dota', 'dada', DAD_SET) for a in ('--read', s, arm, ev / s / K5)]
NAME = constants.PROTO_TABLE_NAME
run([sys.executable, '-m', 'core.tools.zs_arm_table', '--control', CTRL, '--seed', SEED, *reads, '--name', NAME, '--out-dir', local])
diag = local / constants.PROTO_DAD_DIAG_DIRNAME
run([sys.executable, '-m', 'core.tools.dad_position_diag',
     *[x for arm, ev in evs[:-1] for x in ('--row', arm, ev / DAD_SET / K5 / f's{SEED}')],
     '--control', CTRL, '--seed', SEED, '--out-dir', diag])
parts = [f'Control: {CTRL} ({CTRL_ROOT})', (local / f'{constants.PROTO_REPORT_STEM}.md').read_text(),
         '# DoTA-CAP-dev, DADA, DAD: macro and micro\n\n' + (local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md').read_text(),
         '# DAD position diagnostics (control ' + CTRL + ')\n\n' + (diag / 'dad_position_diag.md').read_text()]
(local / 'k0dfi_proto_summary.md').write_text('\n\n'.join(parts))
publish(local, REPORTS)
print((REPORTS / 'k0dfi_proto_summary.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k0dfi_proto_summary.md to Claude')